# Initial Customer Data Generator

This notebook generates the initial synthetic customer snapshot for the CRM source system.

The generated data will later be ingested incrementally into the Bronze layer and updated through CDC events.

In [0]:
from pyspark.sql import functions as F

NUM_CUSTOMERS = 5000
CUSTOMERS_PATH = (
    "/Volumes/workspace/"
    "revenue_leakage_bronze/"
    "landing/crm/customers"
)

customers_base_df = (
    spark.range(1, NUM_CUSTOMERS + 1)
    .withColumn(
        "customer_id",
        F.format_string("C%06d", F.col("id"))
    )
    .select("customer_id")
)

display(customers_base_df.limit(10))

customer_count = customers_base_df.count()
print(f"Number of customers: {customer_count:,}")

In [0]:
first_names = [
    "Aron", "Sara", "Leon", "Emma", "Noah",
    "Mia", "Liam", "Sofia", "David", "Elena"
]

last_names = [
    "Anderson", "Brown", "Davis", "Garcia", "Johnson",
    "Miller", "Martinez", "Taylor", "Wilson", "Clark"
]

first_names_array = F.array(
    *[F.lit(name) for name in first_names]
)

last_names_array = F.array(
    *[F.lit(name) for name in last_names]
)

customers_profile_df = (
    customers_base_df
    .withColumn(
        "customer_number",
        F.substring("customer_id", 2, 6).cast("int")
    )
    .withColumn(
        "first_name",
        F.element_at(
            first_names_array,
            (
                F.pmod(
                    F.col("customer_number") - 1,
                    F.lit(len(first_names))
                ) + 1
            ).cast("int")
        )
    )
    .withColumn(
        "last_name",
        F.element_at(
            last_names_array,
            (
                F.pmod(
                    F.col("customer_number") * 7 - 1,
                    F.lit(len(last_names))
                ) + 1
            ).cast("int")
        )
    )
    .withColumn(
        "email",
        F.lower(
            F.concat(
                F.col("first_name"),
                F.lit("."),
                F.col("last_name"),
                F.col("customer_number"),
                F.lit("@example.com")
            )
        )
    )
)

display(customers_profile_df.limit(10))

In [0]:
locations = [
    ("United States", "California"),
    ("United States", "Texas"),
    ("United States", "New York"),
    ("United States", "Florida"),
    ("Germany", "Berlin"),
    ("Germany", "Bavaria"),
    ("United Kingdom", "England"),
    ("Canada", "Ontario"),
    ("France", "Ile-de-France"),
    ("Netherlands", "North Holland")
]

locations_array = F.array(
    *[
        F.struct(
            F.lit(country).alias("country"),
            F.lit(region).alias("region")
        )
        for country, region in locations
    ]
)

location_index = (
    F.pmod(
        F.col("customer_number") * 3 - 1,
        F.lit(len(locations))
    ) + 1
).cast("int")

segment_bucket = F.pmod(
    F.col("customer_number") * 17 + 3,
    F.lit(20)
)

status_bucket = F.pmod(
    F.col("customer_number") * 13 + 7,
    F.lit(20)
)

signup_day_offset = F.pmod(
    F.col("customer_number") * 37,
    F.lit(1095)
).cast("int")

customers_initial_df = (
    customers_profile_df
    .withColumn(
        "location",
        F.element_at(locations_array, location_index)
    )
    .withColumn(
        "country",
        F.col("location.country")
    )
    .withColumn(
        "region",
        F.col("location.region")
    )
    .withColumn(
        "customer_segment",
        F.when(segment_bucket == 0, F.lit("VIP"))
        .when(segment_bucket <= 5, F.lit("Premium"))
        .otherwise(F.lit("Standard"))
    )
    .withColumn(
        "signup_date",
        F.date_add(
            F.lit("2023-01-01").cast("date"),
            signup_day_offset
        )
    )
    .withColumn(
        "customer_status",
        F.when(status_bucket == 0, F.lit("Inactive"))
        .otherwise(F.lit("Active"))
    )
    .withColumn(
        "operation",
        F.lit("INSERT")
    )
    .withColumn(
        "event_timestamp",
        F.col("signup_date").cast("timestamp")
    )
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "country",
        "region",
        "customer_segment",
        "signup_date",
        "customer_status",
        "operation",
        "event_timestamp"
    )
)

display(customers_initial_df.limit(10))
customers_initial_df.printSchema()

In [0]:
display(
    spark.sql("""
        SELECT
            volume_catalog,
            volume_schema,
            volume_name
        FROM system.information_schema.volumes
        ORDER BY volume_catalog, volume_schema, volume_name
    """)
)

In [0]:
customers_initial_path = (
    "/Volumes/workspace/revenue_leakage_bronze/"
    "landing/customers/initial_load"
)

(
    customers_initial_df.write
    .format("json")
    .mode("overwrite")
    .save(customers_initial_path)
)

# Kontrollojmë a janë ruajtur saktë
saved_customers_df = spark.read.json(customers_initial_path)

print(f"Saved rows: {saved_customers_df.count()}")
display(saved_customers_df)